# Definitivo — analisi e modelli incendi

Confronto temporale isolato con embargo di 7 giorni: regressione logistica, polinomiale, SVM approssimata, albero e Random Forest.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, brier_score_loss, confusion_matrix

RANDOM_STATE = 42
HORIZON = 7
INPUT = Path("output_definitivo/incendi_daily_segmentato_sample_v3.csv")
OUTPUT_DIR = Path("output_definitivo/modelli")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_TRAIN_ROWS = 200_000
if not INPUT.exists(): raise FileNotFoundError(INPUT.resolve())

df = pd.read_csv(INPUT, parse_dates=["date"])
keys = ["segment_id", "lat_cell", "lon_cell"]
df = df.sort_values(keys + ["date"], ignore_index=True)
g = df.groupby(keys, sort=False)

def hist_roll(column, window, agg):
    previous = g[column].shift(1)
    return previous.groupby([df[k] for k in keys], sort=False).transform(lambda x: x.rolling(window, min_periods=window).agg(agg))

for lag in [1, 3, 7, 14]:
    df[f"detection_lag_{lag}d"] = g["detection_count"].shift(lag)
for window in [3, 7, 14, 30]:
    df[f"detection_sum_last_{window}d"] = hist_roll("detection_count", window, "sum")
    df[f"active_days_last_{window}d"] = hist_roll("active_fire_day", window, "sum")
    df[f"frp_sum_last_{window}d"] = hist_roll("daily_frp_sum", window, "sum")
df["frp_mean_active_last_7d"] = hist_roll("daily_frp_sum", 7, "sum") / df["active_days_last_7d"].replace(0, np.nan)
df["day_of_year"] = df.date.dt.dayofyear.astype("int16")
df["sin_doy"] = np.sin(2*np.pi*df.day_of_year/365.25)
df["cos_doy"] = np.cos(2*np.pi*df.day_of_year/365.25)

future = pd.concat([g["detection_count"].shift(-i).rename(i) for i in range(1, HORIZON+1)], axis=1)
count = future.sum(axis=1, min_count=HORIZON)
df["fire_count_next_7d"] = count
df["fire_next_7d"] = pd.Series(np.where(count.notna(), (count > 0).astype("int8"), pd.NA), index=df.index, dtype="Int8")
df["target_horizon_complete_7d"] = count.notna().astype("int8")

features = ["detection_lag_1d", "detection_lag_3d", "detection_lag_7d", "detection_lag_14d", "detection_sum_last_3d", "detection_sum_last_7d", "detection_sum_last_14d", "detection_sum_last_30d", "active_days_last_3d", "active_days_last_7d", "active_days_last_14d", "active_days_last_30d", "frp_sum_last_7d", "frp_sum_last_14d", "frp_mean_active_last_7d", "sin_doy", "cos_doy"]
ml = df.dropna(subset=features + ["fire_next_7d"]).copy()
ml["fire_next_7d"] = ml["fire_next_7d"].astype(int)
ml[keys + ["date"] + features + ["fire_next_7d", "fire_count_next_7d", "target_horizon_complete_7d"]].to_csv(OUTPUT_DIR / "dati_ml_storico_v3.csv", index=False)

# Split esclusivamente temporale nel segmento piu lungo; 7 giorni di embargo separano train, validation e test.
segment = int(ml.groupby("segment_id").date.nunique().idxmax())
work = ml[ml.segment_id.eq(segment)].copy()
dates = np.array(sorted(work.date.unique()))
val_days, test_days, embargo = 21, 21, HORIZON
train_end = len(dates) - val_days - test_days - 2*embargo
if train_end < 30: raise RuntimeError("Segmento troppo corto per split con embargo")
splits = {"train": dates[:train_end], "validation": dates[train_end+embargo:train_end+embargo+val_days], "test": dates[train_end+embargo+val_days+embargo:]}
train, val, test = (work[work.date.isin(splits[n])].copy() for n in ["train", "validation", "test"])

def balanced_sample(frame):
    if len(frame) <= MAX_TRAIN_ROWS: return frame
    sampled=[]
    for label, group in frame.groupby("fire_next_7d", observed=True):
        size=min(len(group), max(1, int(MAX_TRAIN_ROWS * len(group) / len(frame))))
        sampled.append(group.sample(n=size, random_state=RANDOM_STATE))
    return pd.concat(sampled, ignore_index=True)
train_fit = balanced_sample(train)

def metrics(y, p, threshold):
    pred = p >= threshold; tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {"accuracy":accuracy_score(y,pred), "balanced_accuracy":balanced_accuracy_score(y,pred), "precision":precision_score(y,pred,zero_division=0), "recall":recall_score(y,pred,zero_division=0), "f1":f1_score(y,pred,zero_division=0), "roc_auc":roc_auc_score(y,p), "pr_auc":average_precision_score(y,p), "mcc":matthews_corrcoef(y,pred), "brier":brier_score_loss(y,p), "specificity":tn/max(tn+fp,1), "false_positive_rate":fp/max(tn+fp,1), "false_negative_rate":fn/max(fn+tp,1)}

def choose_threshold(y, p):
    grid=np.linspace(.05,.95,91); scores=[f1_score(y,p>=t,zero_division=0) for t in grid]
    return float(grid[int(np.argmax(scores))])

models = {
 "regressione_logistica": Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE))]),
 "regressione_polinomiale": Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler()),("poly",PolynomialFeatures(2,include_bias=False)),("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE))]),
 "svm_rbf_approssimata": Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler()),("rbf",RBFSampler(gamma=0.05,n_components=400,random_state=RANDOM_STATE)),("model",LogisticRegression(max_iter=700,class_weight="balanced",random_state=RANDOM_STATE))]),
 "albero_decisionale": Pipeline([("impute",SimpleImputer(strategy="median")),("model",DecisionTreeClassifier(max_depth=10,min_samples_leaf=150,class_weight="balanced",random_state=RANDOM_STATE))]),
 "random_forest": Pipeline([("impute",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=120,max_depth=16,min_samples_leaf=100,class_weight="balanced_subsample",n_jobs=-1,random_state=RANDOM_STATE))]),
}
rows=[]
for name, model in models.items():
    start=time.perf_counter(); model.fit(train_fit[features],train_fit.fire_next_7d)
    pv=model.predict_proba(val[features])[:,1]; threshold=choose_threshold(val.fire_next_7d,pv)
    pt=model.predict_proba(test[features])[:,1]
    row={"model":name,"target":"fire_next_7d","segment_id":segment,"threshold_validation":threshold,"train_rows":len(train_fit),"validation_rows":len(val),"test_rows":len(test),"train_end":str(train.date.max().date()),"validation_period":f"{val.date.min().date()}:{val.date.max().date()}","test_period":f"{test.date.min().date()}:{test.date.max().date()}","training_seconds":time.perf_counter()-start,"evaluation":"test_temporale_isolato_con_embargo_7d"}
    row.update(metrics(test.fire_next_7d,pt,threshold)); rows.append(row)
report=pd.DataFrame(rows).sort_values(["pr_auc","f1"],ascending=False)
report.to_csv(OUTPUT_DIR / "confronto_modelli_storico_v3.csv",index=False)
pd.DataFrame([{"segment":segment,"train_days":len(splits['train']),"validation_days":len(splits['validation']),"test_days":len(splits['test']),"embargo_days":embargo,"prevalence_train":train.fire_next_7d.mean(),"prevalence_validation":val.fire_next_7d.mean(),"prevalence_test":test.fire_next_7d.mean()}]).to_csv(OUTPUT_DIR / "split_temporale_storico_v3.csv",index=False)
with open(OUTPUT_DIR / "configurazione_modelli_v3.json","w") as f: json.dump({"random_state":RANDOM_STATE,"horizon":HORIZON,"features":features,"max_train_rows":MAX_TRAIN_ROWS},f,indent=2)
print(report[["model","roc_auc","pr_auc","precision","recall","f1","threshold_validation"]].to_string(index=False))


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

GRAPH_DIR = Path("output_definitivo/modelli/grafici")
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

# 1. Confronto sintetico dei modelli.
plot_report = report.melt(id_vars="model", value_vars=["roc_auc", "pr_auc", "f1"], var_name="metrica", value_name="valore")
plt.figure(figsize=(11, 5)); sns.barplot(data=plot_report, x="model", y="valore", hue="metrica")
plt.ylim(0, 1); plt.xticks(rotation=20, ha="right"); plt.title("Confronto modelli — test temporale isolato")
plt.tight_layout(); plt.savefig(GRAPH_DIR / "01_confronto_modelli.png", dpi=160); plt.close()

# 2. Media mobile a 3 giorni del tasso di target positivo per segmento.
trend = ml.groupby(["segment_id", "date"], as_index=False).fire_next_7d.mean().sort_values(["segment_id", "date"])
trend["media_mobile_3d"] = trend.groupby("segment_id").fire_next_7d.transform(lambda s: s.rolling(3, min_periods=1).mean())
plt.figure(figsize=(12, 5)); sns.lineplot(data=trend, x="date", y="media_mobile_3d", hue="segment_id", palette="tab10")
plt.ylim(0, 1); plt.title("Target fire_next_7d — media mobile 3 giorni")
plt.tight_layout(); plt.savefig(GRAPH_DIR / "02_target_media_mobile_3_giorni.png", dpi=160); plt.close()

# 3. Bootstrap Monte Carlo per cella sul test: intervallo della prevalenza del target.
test_cells = test[["lat_cell", "lon_cell", "fire_next_7d"]].groupby(["lat_cell", "lon_cell"], observed=True).fire_next_7d.agg(["mean", "count"])
rng=np.random.default_rng(RANDOM_STATE); means=test_cells["mean"].to_numpy(); counts=test_cells["count"].to_numpy()
idx=rng.integers(0,len(test_cells),size=(500,len(test_cells))); estimates=(means[idx]*counts[idx]).sum(axis=1)/counts[idx].sum(axis=1)
plt.figure(figsize=(9,5)); sns.histplot(estimates, bins=30, color="#1f77b4")
plt.axvline(np.quantile(estimates,.025), color="black", linestyle="--"); plt.axvline(np.quantile(estimates,.975), color="black", linestyle="--")
plt.title("Monte Carlo bootstrap per cella — prevalenza test")
plt.tight_layout(); plt.savefig(GRAPH_DIR / "03_montecarlo_bootstrap.png", dpi=160); plt.close()
pd.DataFrame({"estimate":estimates}).to_csv(GRAPH_DIR / "montecarlo_bootstrap.csv",index=False)

# 4. Correlazioni delle feature, calcolate su un campione riproducibile.
corr_data = ml[features].sample(n=min(100_000,len(ml)), random_state=RANDOM_STATE).corr()
plt.figure(figsize=(14,11)); sns.heatmap(corr_data, cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Matrice di correlazione feature")
plt.tight_layout(); plt.savefig(GRAPH_DIR / "04_matrice_correlazione.png", dpi=160); plt.close(); corr_data.to_csv(GRAPH_DIR / "matrice_correlazione.csv")

# 5. Matrice di confusione della Random Forest al threshold scelto solo in validation.
rf = models["random_forest"]
rf_threshold = float(report.loc[report.model.eq("random_forest"), "threshold_validation"].iloc[0])
rf_pred = (rf.predict_proba(test[features])[:,1] >= rf_threshold).astype(int)
fig, ax = plt.subplots(figsize=(5,5)); ConfusionMatrixDisplay.from_predictions(test.fire_next_7d, rf_pred, ax=ax, colorbar=False)
ax.set_title("Random Forest — test temporale isolato")
fig.tight_layout(); fig.savefig(GRAPH_DIR / "05_matrice_confusione_random_forest.png", dpi=160); plt.close(fig)
print({"grafici": 5, "directory": str(GRAPH_DIR), "bootstrap_ci95": [float(np.quantile(estimates,.025)), float(np.quantile(estimates,.975))]})